In [1]:
!pip install tensorflow_hub resampy soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 21.7 MB/s eta 0:00:00


In [6]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import csv
import resampy
import soundfile as sf
from IPython.display import Audio, display

# YAMNet 모델 로드 (1~2분 걸림)
print("모델 로드 중...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

# 클래스 이름 로드
class_map_path = yamnet_model.class_map_path().numpy().decode('utf-8')
with tf.io.gfile.GFile(class_map_path) as f:
    class_names = [row['display_name'] for row in csv.DictReader(f)]

print(f"✅ 모델 로드 완료! 총 {len(class_names)}개 클래스 인식 가능")

모델 로드 중...
✅ 모델 로드 완료! 총 521개 클래스 인식 가능


In [7]:
# 각 조원의 담당 소리 키워드 설정
SOUND_CATEGORIES = {
    '화재벨':     ['fire', 'alarm', 'siren', 'smoke'],
    '아기울음':   ['baby', 'cry', 'infant'],
    '개소리':     ['dog', 'bark', 'howl', 'growl'],
    '초인종':     ['doorbell', 'ding-dong', 'bell'],
    '현관문':     ['door', 'knock', 'slam'],
    '물소리':     ['water', 'faucet', 'tap', 'drip', 'pour'],
}

def load_and_classify(file_path):
    # 오디오 로드
    wav_data, sample_rate = sf.read(file_path, dtype=np.int16)
    if len(wav_data.shape) > 1:
        wav_data = wav_data.mean(axis=1)
    waveform = wav_data.astype(np.float32) / 32768.0
    if sample_rate != 16000:
        waveform = resampy.resample(waveform, sample_rate, 16000)

    # YAMNet 추론
    scores, embeddings, spectrogram = yamnet_model(waveform)
    mean_scores = scores.numpy().mean(axis=0)
    top_indices = np.argsort(mean_scores)[::-1][:10]

    # 오디오 재생
    print(f"\n🎵 파일: {file_path} ({len(waveform)/16000:.1f}초)")
    display(Audio(waveform, rate=16000))

    # Top-10 결과 출력
    print("\n📊 Top-10 분류 결과:")
    for rank, i in enumerate(top_indices, 1):
        name = class_names[i]
        score = mean_scores[i]
        matched = ""
        for category, keywords in SOUND_CATEGORIES.items():
            if any(kw in name.lower() for kw in keywords):
                matched = f" ← {category}"
                break
        print(f"  {rank:2d}. {name:<35s} {score:.4f}{matched}")

    # 카테고리별 점수 요약
    print("\n📋 카테고리별 점수:")
    cat_totals = {}
    for category, keywords in SOUND_CATEGORIES.items():
        cat_totals[category] = sum(
            mean_scores[i] for i in range(len(class_names))
            if any(kw in class_names[i].lower() for kw in keywords)
        )
        bar = "█" * int(cat_totals[category] * 50)
        print(f"  {category:<8s} {cat_totals[category]:.4f} {bar}")

    # ===== 여기가 추가된 부분 =====
    # 최종 판정: 가장 높은 카테고리 찾기
    best_cat = max(cat_totals, key=cat_totals.get)
    best_score = cat_totals[best_cat]

    print(f"\n🏷️ 최종 판정: {best_cat} (총점: {best_score:.4f})")
    if best_score >= 0.1:
        print(f"  ✅ '{best_cat}'(으)로 인식 성공!")
    else:
        print(f"  ❌ 어떤 카테고리에도 명확히 매칭되지 않음 - fine-tuning 필요")

print("✅ 분류 함수 준비 완료! (전체 카테고리 지원)")

✅ 분류 함수 준비 완료! (전체 카테고리 지원)


In [4]:
from google.colab import files

print("=" * 50)
print("🎧 소리 파일을 업로드하세요!")
print("   wav, mp3, flac 다 가능해요")
print("   여러 파일 한 번에 선택 가능!")
print("=" * 50)

uploaded = files.upload()

for filename in uploaded:
    load_and_classify(filename)

print("\n" + "=" * 50)
print(f"📋 총 {len(uploaded)}개 파일 테스트 완료!")
print("=" * 50)

🎧 소리 파일을 업로드하세요!
   wav, mp3, flac 다 가능해요
   여러 파일 한 번에 선택 가능!


Saving 0297_D_005.wav to 0297_D_005.wav

🎵 파일: 0297_D_005.wav (1.7초)



📊 Top-10 분류 결과:
   1. Frog                                0.3159
   2. Crying, sobbing                     0.2550 ← 아기울음
   3. Baby cry, infant cry                0.1782 ← 아기울음
   4. Animal                              0.1713
   5. Screaming                           0.0959
   6. Owl                                 0.0901
   7. Croak                               0.0860
   8. Wild animals                        0.0674
   9. Buzz                                0.0610
  10. Whimper                             0.0535

📋 카테고리별 점수:
  화재벨      0.0316 █
  아기울음     0.4333 █████████████████████
  개소리      0.0004 
  초인종      0.0008 
  현관문      0.0000 
  물소리      0.0005 

📋 총 1개 파일 테스트 완료!


In [8]:
from google.colab import files

print("=" * 50)
print("🎧 소리 파일을 업로드하세요!")
print("   wav, mp3, flac 다 가능해요")
print("   여러 파일 한 번에 선택 가능!")
print("=" * 50)

uploaded = files.upload()

for filename in uploaded:
    load_and_classify(filename)

print("\n" + "=" * 50)
print(f"📋 총 {len(uploaded)}개 파일 테스트 완료!")
print("=" * 50)

🎧 소리 파일을 업로드하세요!
   wav, mp3, flac 다 가능해요
   여러 파일 한 번에 선택 가능!


Saving husky_train_00108.wav to husky_train_00108 (1).wav
Saving husky_train_00274.wav to husky_train_00274 (1).wav
Saving husky_train_00392.wav to husky_train_00392 (1).wav
Saving husky_train_00408.wav to husky_train_00408 (1).wav
Saving husky_train_00468.wav to husky_train_00468 (1).wav
Saving shiba_train_00319.wav to shiba_train_00319 (1).wav
Saving shiba_train_00328.wav to shiba_train_00328 (1).wav
Saving shiba_train_00336.wav to shiba_train_00336 (1).wav
Saving shiba_train_00342.wav to shiba_train_00342 (1).wav
Saving shiba_train_00488.wav to shiba_train_00488 (1).wav

🎵 파일: husky_train_00108 (1).wav (8.7초)



📊 Top-10 분류 결과:
   1. Dog                                 0.5835 ← 개소리
   2. Domestic animals, pets              0.5747
   3. Animal                              0.5504
   4. Howl                                0.3265 ← 개소리
   5. Canidae, dogs, wolves               0.2533 ← 개소리
   6. Yip                                 0.2307
   7. Whimper (dog)                       0.1743 ← 개소리
   8. Bow-wow                             0.1154
   9. Bark                                0.0951 ← 개소리
  10. Wild animals                        0.0822

📋 카테고리별 점수:
  화재벨      0.0036 
  아기울음     0.0270 █
  개소리      1.4876 ██████████████████████████████████████████████████████████████████████████
  초인종      0.0005 
  현관문      0.0000 
  물소리      0.0001 

🏷️ 최종 판정: 개소리 (총점: 1.4876)
  ✅ '개소리'(으)로 인식 성공!

🎵 파일: husky_train_00274 (1).wav (12.5초)



📊 Top-10 분류 결과:
   1. Domestic animals, pets              0.5231
   2. Dog                                 0.5195 ← 개소리
   3. Animal                              0.4693
   4. Speech                              0.2352
   5. Howl                                0.2209 ← 개소리
   6. Canidae, dogs, wolves               0.1712 ← 개소리
   7. Yip                                 0.1622
   8. Bark                                0.1498 ← 개소리
   9. Bow-wow                             0.0885
  10. Whimper (dog)                       0.0701 ← 개소리

📋 카테고리별 점수:
  화재벨      0.0178 
  아기울음     0.0185 
  개소리      1.1327 ████████████████████████████████████████████████████████
  초인종      0.0014 
  현관문      0.0003 
  물소리      0.0001 

🏷️ 최종 판정: 개소리 (총점: 1.1327)
  ✅ '개소리'(으)로 인식 성공!

🎵 파일: husky_train_00392 (1).wav (9.1초)



📊 Top-10 분류 결과:
   1. Animal                              0.8623
   2. Domestic animals, pets              0.8346
   3. Dog                                 0.8176 ← 개소리
   4. Howl                                0.7060 ← 개소리
   5. Canidae, dogs, wolves               0.4862 ← 개소리
   6. Yip                                 0.3467
   7. Whimper (dog)                       0.2227 ← 개소리
   8. Wild animals                        0.1961
   9. Bow-wow                             0.0990
  10. Bark                                0.0655 ← 개소리

📋 카테고리별 점수:
  화재벨      0.0009 
  아기울음     0.0105 
  개소리      2.3001 ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  초인종      0.0000 
  현관문      0.0000 
  물소리      0.0000 

🏷️ 최종 판정: 개소리 (총점: 2.3001)
  ✅ '개소리'(으)로 인식 성공!

🎵 파일: husky_train_00408 (1).wav (30.0초)


...


📊 Top-10 분류 결과:
   1. Animal                              0.4358
   2. Dog                                 0.4236 ← 개소리
   3. Domestic animals, pets              0.4096
   4. Howl                                0.1879 ← 개소리
   5. Bow-wow                             0.1713
   6. Yip                                 0.1712
   7. Canidae, dogs, wolves               0.1651 ← 개소리
   8. Speech                              0.1250
   9. Bark                                0.1106 ← 개소리
  10. Whimper (dog)                       0.1058 ← 개소리

📋 카테고리별 점수:
  화재벨      0.0165 
  아기울음     0.0606 ███
  개소리      0.9991 █████████████████████████████████████████████████
  초인종      0.0119 
  현관문      0.0032 
  물소리      0.0003 

🏷️ 최종 판정: 개소리 (총점: 0.9991)
  ✅ '개소리'(으)로 인식 성공!

🎵 파일: shiba_train_00328 (1).wav (18.0초)



📊 Top-10 분류 결과:
   1. Domestic animals, pets              0.3667
   2. Animal                              0.3628
   3. Dog                                 0.3396 ← 개소리
   4. Crying, sobbing                     0.1239 ← 아기울음
   5. Bark                                0.1212 ← 개소리
   6. Whimper                             0.0904
   7. Canidae, dogs, wolves               0.0874 ← 개소리
   8. Yip                                 0.0827
   9. Howl                                0.0719 ← 개소리
  10. Baby cry, infant cry                0.0682 ← 아기울음

📋 카테고리별 점수:
  화재벨      0.0040 
  아기울음     0.1923 █████████
  개소리      0.6611 █████████████████████████████████
  초인종      0.0016 
  현관문      0.0005 
  물소리      0.0000 

🏷️ 최종 판정: 개소리 (총점: 0.6611)
  ✅ '개소리'(으)로 인식 성공!

🎵 파일: shiba_train_00336 (1).wav (30.0초)



📊 Top-10 분류 결과:
   1. Animal                              0.4590
   2. Domestic animals, pets              0.3366
   3. Dog                                 0.3346 ← 개소리
   4. Speech                              0.0882
   5. Pig                                 0.0814
   6. Whimper (dog)                       0.0697 ← 개소리
   7. Livestock, farm animals, working animals 0.0674
   8. Canidae, dogs, wolves               0.0611 ← 개소리
   9. Oink                                0.0439
  10. Yip                                 0.0431

📋 카테고리별 점수:
  화재벨      0.0077 
  아기울음     0.0010 
  개소리      0.4928 ████████████████████████
  초인종      0.0021 
  현관문      0.0021 
  물소리      0.0352 █

🏷️ 최종 판정: 개소리 (총점: 0.4928)
  ✅ '개소리'(으)로 인식 성공!

🎵 파일: shiba_train_00342 (1).wav (16.8초)



📊 Top-10 분류 결과:
   1. Animal                              0.3890
   2. Domestic animals, pets              0.3549
   3. Dog                                 0.3451 ← 개소리
   4. Crying, sobbing                     0.2471 ← 아기울음
   5. Baby cry, infant cry                0.1686 ← 아기울음
   6. Whimper                             0.1628
   7. Speech                              0.1187
   8. Yip                                 0.1111
   9. Canidae, dogs, wolves               0.0986 ← 개소리
  10. Whimper (dog)                       0.0838 ← 개소리

📋 카테고리별 점수:
  화재벨      0.0049 
  아기울음     0.4192 ████████████████████
  개소리      0.6810 ██████████████████████████████████
  초인종      0.0041 
  현관문      0.0001 
  물소리      0.0002 

🏷️ 최종 판정: 개소리 (총점: 0.6810)
  ✅ '개소리'(으)로 인식 성공!

🎵 파일: shiba_train_00488 (1).wav (12.6초)



📊 Top-10 분류 결과:
   1. Animal                              0.4513
   2. Domestic animals, pets              0.3425
   3. Dog                                 0.2530 ← 개소리
   4. Livestock, farm animals, working animals 0.1255
   5. Chicken, rooster                    0.1131
   6. Fowl                                0.1060
   7. Squeak                              0.0960
   8. Cluck                               0.0724
   9. Wild animals                        0.0716
  10. Bird                                0.0673

📋 카테고리별 점수:
  화재벨      0.0070 
  아기울음     0.0129 
  개소리      0.4136 ████████████████████
  초인종      0.0032 
  현관문      0.0241 █
  물소리      0.0003 

🏷️ 최종 판정: 개소리 (총점: 0.4136)
  ✅ '개소리'(으)로 인식 성공!

📋 총 10개 파일 테스트 완료!
